In [1]:
# ===============================
# 1. 라이브러리
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    classification_report, roc_auc_score, f1_score,
    precision_recall_fscore_support, average_precision_score,
    precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from lightgbm import LGBMClassifier

In [2]:
# ===============================
# 2. 데이터 로드
# ===============================
df = pd.read_csv("../data/insurance_policyholder_churn_synthetic.csv")

In [3]:
# ===============================
# 3. 데이터 전처리 + 피처 엔지니어링
# ===============================
def feature_engineering(df):
    """
    LightGBM 성능을 높이는 파생 피처 추가.
    데이터셋 컬럼에 맞게 조건부로 생성.
    """
    df = df.copy()

    # 예시: 숫자형 컬럼 간 비율/interaction (실제 컬럼명에 맞게 수정)
    # 보험 데이터에서 흔히 유효한 피처들
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    num_cols = [c for c in num_cols if c not in
                ['customer_id', 'churn_flag', 'churn_probability_true']]

    # 예: premium 대비 claim 비율 (컬럼 존재 시)
    if 'premium_amount' in df.columns and 'claim_amount' in df.columns:
        df['claim_to_premium_ratio'] = (
            df['claim_amount'] / (df['premium_amount'] + 1)
        )

    # 예: tenure 구간화 (컬럼 존재 시)
    if 'tenure_months' in df.columns:
        df['tenure_bucket'] = pd.cut(
            df['tenure_months'],
            bins=[0, 12, 36, 60, 120, 9999],
            labels=['<1yr', '1-3yr', '3-5yr', '5-10yr', '10yr+']
        ).astype(str)

    # 예: 최근 민원/컨택 여부 (컬럼 존재 시)
    if 'months_since_last_contact' in df.columns:
        df['is_long_inactive'] = (
            df['months_since_last_contact'] > 12
        ).astype(int)

    return df


def preprocess_data(df):
    df = feature_engineering(df)

    y = df['churn_flag']

    drop_cols = [
        'customer_id', 'as_of_date', 'churn_flag',
        'churn_type', 'churn_probability_true'
    ]
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])

    return X, y


X, y = preprocess_data(df)

print(f"클래스 분포:\n{y.value_counts()}")
print(f"Churn 비율: {y.mean():.3f}")

클래스 분포:
churn_flag
0    34917
1    15083
Name: count, dtype: int64
Churn 비율: 0.302


In [4]:
# ===============================
# 4. Train/Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
# ===============================
# 5. 컬럼 구분
# ===============================
numeric_features   = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()


In [6]:
# ===============================
# 6. 전처리 파이프라인
# ===============================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [7]:
# ===============================
# 7. 클래스 불균형 비율 계산
# ===============================
# class_weight를 고정("balanced")하지 않고 탐색 범위에 포함
churn_ratio = y_train.value_counts()[0] / y_train.value_counts()[1]
print(f"비churn:churn 비율 = {churn_ratio:.1f}:1")

# scale_pos_weight 후보: 실제 비율 기준으로 ±범위 탐색
spw_low  = max(1.0, churn_ratio * 0.5)
spw_mid  = churn_ratio
spw_high = churn_ratio * 2.0

비churn:churn 비율 = 2.3:1


In [8]:
# ===============================
# 8. 커스텀 Scorer: F-beta (beta=2)
#    → recall을 2배 중시하되 precision도 반영
# ===============================
from sklearn.metrics import make_scorer, fbeta_score

# beta=2: recall 가중 / beta=1: 일반 F1
BETA = 2
fbeta_scorer = make_scorer(fbeta_score, beta=BETA, pos_label=1)

In [9]:
# ===============================
# 9. 하이퍼파라미터 탐색 (보완)
# ===============================
base_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LGBMClassifier(random_state=42))
])

param_dist = {
    # 기존 파라미터
    "clf__n_estimators":      [200, 300, 500, 700, 1000],
    "clf__learning_rate":     [0.01, 0.03, 0.05, 0.1],
    "clf__max_depth":         [3, 5, 7, 10, -1],
    "clf__num_leaves":        [15, 31, 63, 127],
    "clf__subsample":         [0.6, 0.7, 0.8, 0.9],
    "clf__colsample_bytree":  [0.6, 0.7, 0.8, 0.9],
    "clf__min_child_samples": [10, 20, 30, 50],

    # ★ 추가: 클래스 불균형 처리를 파라미터로 탐색
    # scale_pos_weight: 양성(churn) 클래스에 부여하는 가중치
    "clf__scale_pos_weight":  [spw_low, spw_mid, spw_high],

    # ★ 추가: 정규화 강도 (과적합 방지 → precision 안정)
    "clf__reg_alpha":         [0.0, 0.01, 0.1, 0.5],   # L1
    "clf__reg_lambda":        [0.0, 0.1, 1.0, 5.0],    # L2

    # ★ 추가: 최소 gain (작은 트리 분기 억제)
    "clf__min_split_gain":    [0.0, 0.01, 0.05, 0.1],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=40,               # ★ 20 → 40 (파라미터 공간이 넓어졌으므로)
    scoring=fbeta_scorer,    # ★ "recall" → F-beta(β=2)
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    refit=True
)

print("튜닝 시작...")
search.fit(X_train, y_train)

print("\n===== 튜닝 결과 =====")
print("Best Params:", search.best_params_)
print(f"Best CV F{BETA}:", search.best_score_)

튜닝 시작...
Fitting 5 folds for each of 40 candidates, totalling 200 fits
[LightGBM] [Info] Number of positive: 12066, number of negative: 27934
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012939 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2747
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 53
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.301650 -> initscore=-0.839453
[LightGBM] [Info] Start training from score -0.839453
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [10]:
# ===============================
# 10. 최적 모델
# ===============================
best_model = search.best_estimator_

In [11]:
# ===============================
# 11. 기본 성능 평가 (threshold=0.5)
# ===============================
y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("\n===== 기본 성능 (threshold=0.50) =====")
print(classification_report(y_test, y_pred))
print("ROC AUC:       ", roc_auc_score(y_test, y_prob))
print("PR AUC (AP):   ", average_precision_score(y_test, y_prob))  # ★ 추가
print("F1:            ", f1_score(y_test, y_pred))


===== 기본 성능 (threshold=0.50) =====
              precision    recall  f1-score   support

           0       0.91      0.43      0.59      6983
           1       0.41      0.90      0.56      3017

    accuracy                           0.57     10000
   macro avg       0.66      0.67      0.57     10000
weighted avg       0.76      0.57      0.58     10000

ROC AUC:        0.7915782355282925
PR AUC (AP):    0.6639927838459695
F1:             0.5614758322168402


C:\Users\Playdata\miniconda3\envs\tp_returns\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [12]:
# ===============================
# 12. ★ Threshold 탐색 (개선된 선택 기준)
# ===============================
# 전략: recall >= MIN_RECALL 이면서 precision >= MIN_PRECISION인 구간에서
#       F-beta가 가장 높은 threshold 선택

MIN_RECALL    = 0.70   # recall 하한
MIN_PRECISION = 0.30   # ★ precision 하한 명시 (너무 낮아지는 것 방지)

print("\n===== Threshold Search =====")
print(f"기준: recall >= {MIN_RECALL}, precision >= {MIN_PRECISION}")
print(f"{'threshold':>10} | {'precision':>10} | {'recall':>8} | {'f1':>8} | {'fbeta':>8} | {'선택'}")
print("-" * 65)

best_threshold = 0.5
best_fbeta     = 0.0
threshold_results = []

for t in np.arange(0.10, 0.71, 0.02):   # ★ 간격 0.05 → 0.02 (세밀하게)
    y_pred_t = (y_prob >= t).astype(int)

    p, r, f1, _ = precision_recall_fscore_support(
        y_test, y_pred_t, average=None, zero_division=0
    )

    # 양성(churn=1) 클래스 기준
    p1  = p[1]  if len(p)  > 1 else 0
    r1  = r[1]  if len(r)  > 1 else 0
    f1_ = f1[1] if len(f1) > 1 else 0

    # F-beta 직접 계산 (β=2)
    denom = (BETA**2 * p1 + r1)
    fb = (1 + BETA**2) * p1 * r1 / denom if denom > 0 else 0

    meets = (r1 >= MIN_RECALL) and (p1 >= MIN_PRECISION)
    marker = " ◀" if meets else ""

    threshold_results.append({
        "threshold":   round(t, 2),
        "precision_1": round(p1,  4),
        "recall_1":    round(r1,  4),
        "f1_1":        round(f1_, 4),
        "fbeta_1":     round(fb,  4),
        "meets_criteria": meets
    })

    print(f"{t:>10.2f} | {p1:>10.3f} | {r1:>8.3f} | {f1_:>8.3f} | {fb:>8.3f}{marker}")

    if meets and fb > best_fbeta:
        best_fbeta     = fb
        best_threshold = t

print(f"\n★ 선택된 threshold: {best_threshold:.2f}  (F{BETA}={best_fbeta:.4f})")


===== Threshold Search =====
기준: recall >= 0.7, precision >= 0.3
 threshold |  precision |   recall |       f1 |    fbeta | 선택
-----------------------------------------------------------------
      0.10 |      0.302 |    1.000 |    0.464 |    0.684 ◀
      0.12 |      0.303 |    0.999 |    0.465 |    0.684 ◀
      0.14 |      0.305 |    0.997 |    0.467 |    0.686 ◀
      0.16 |      0.308 |    0.996 |    0.470 |    0.688 ◀
      0.18 |      0.312 |    0.994 |    0.475 |    0.692 ◀
      0.20 |      0.317 |    0.992 |    0.480 |    0.695 ◀
      0.22 |      0.322 |    0.990 |    0.486 |    0.699 ◀
      0.24 |      0.327 |    0.986 |    0.491 |    0.703 ◀
      0.26 |      0.332 |    0.984 |    0.496 |    0.706 ◀
      0.28 |      0.336 |    0.980 |    0.500 |    0.708 ◀
      0.30 |      0.339 |    0.976 |    0.504 |    0.710 ◀
      0.32 |      0.343 |    0.971 |    0.507 |    0.710 ◀
      0.34 |      0.346 |    0.967 |    0.510 |    0.712 ◀
      0.36 |      0.352 |    0.963 |   

In [13]:
# ===============================
# 13. 최종 성능 평가
# ===============================
final_pred = (y_prob >= best_threshold).astype(int)

print("\n===== 최종 성능 =====")
print(classification_report(y_test, final_pred))
print("ROC AUC:     ", roc_auc_score(y_test, y_prob))
print("PR AUC (AP): ", average_precision_score(y_test, y_prob))
print(f"F{BETA}:         ", fbeta_score(y_test, final_pred, beta=BETA))


===== 최종 성능 =====
              precision    recall  f1-score   support

           0       0.91      0.43      0.59      6983
           1       0.41      0.90      0.56      3017

    accuracy                           0.57     10000
   macro avg       0.66      0.67      0.57     10000
weighted avg       0.76      0.57      0.58     10000

ROC AUC:      0.7915782355282925
PR AUC (AP):  0.6639927838459695
F2:          0.7262450677188866


In [14]:
# ===============================
# 14. Threshold 결과표
# ===============================
threshold_df = pd.DataFrame(threshold_results)
print("\n===== Threshold 결과표 =====")
print(threshold_df.to_string(index=False))


===== Threshold 결과표 =====
 threshold  precision_1  recall_1   f1_1  fbeta_1  meets_criteria
      0.10       0.3018    1.0000 0.4637   0.6837            True
      0.12       0.3029    0.9990 0.4648   0.6844            True
      0.14       0.3048    0.9970 0.4669   0.6856            True
      0.16       0.3075    0.9960 0.4700   0.6880            True
      0.18       0.3120    0.9940 0.4750   0.6917            True
      0.20       0.3168    0.9920 0.4802   0.6955            True
      0.22       0.3219    0.9897 0.4858   0.6995            True
      0.24       0.3269    0.9864 0.4910   0.7028            True
      0.26       0.3316    0.9838 0.4960   0.7060            True
      0.28       0.3358    0.9804 0.5003   0.7085            True
      0.30       0.3394    0.9758 0.5037   0.7097            True
      0.32       0.3427    0.9705 0.5065   0.7103            True
      0.34       0.3463    0.9669 0.5100   0.7118            True
      0.36       0.3515    0.9629 0.5150   0.7144